# Expense Classifier
**Goal:** Classify credit card transactions into 14 expense categories using Machine Learning and BERT.

**Dataset:** Kaggle Credit Card Transactions Dataset (50,000 rows)

**Metric:** Macro F1 Score

---

## Phase 1 — Dataset Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset — update path if running on Colab
df = pd.read_csv('/kaggle/input/expense-classifier-data/transactions.csv', nrows=50000)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nLabel distribution:')
print(df['category'].value_counts())

In [ ]:
# Keep only the columns we will use
cols_to_keep = ['merchant', 'category', 'amt', 'gender', 'city_pop', 'job']
df_clean = df[cols_to_keep].copy()

print('Clean dataframe shape:', df_clean.shape)
print('Any nulls?', df_clean.isnull().sum().sum())
df_clean.head()

### Phase 1 Complete
- Dataset: 50,000 rows, 14 categories
- Key input feature: amt (transaction amount)
- Supporting features: job, city_pop, gender
- No nulls, no resampling needed

---

## Phase 2 — Exploratory Data Analysis (EDA)

In [ ]:
# Chart 1: Transaction count per category
plt.figure(figsize=(12, 5))
df_clean['category'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Transaction count per category')
plt.xlabel('Category')
plt.ylabel('Number of transactions')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Transaction amount distribution per category (log scale)
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_clean, x='category', y='amt',
            hue='category', palette='Set2', legend=False)
plt.yscale('log')
plt.title('Transaction amount distribution per category (log scale)')
plt.xlabel('Category')
plt.ylabel('Amount (log scale)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Gender distribution per category
plt.figure(figsize=(14, 5))
gender_cat = df_clean.groupby(['category', 'gender']).size().unstack()
gender_cat.plot(kind='bar', figsize=(14, 5), color=['#FF9999', '#6699CC'], edgecolor='white')
plt.title('Gender distribution per category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Median transaction amount per category
plt.figure(figsize=(12, 5))
df_clean.groupby('category')['amt'].median().sort_values(ascending=False).plot(
    kind='bar', color='teal', edgecolor='white')
plt.title('Median transaction amount per category')
plt.xlabel('Category')
plt.ylabel('Median amount ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Phase 2 Complete — EDA Summary

**Key findings:**
- 14 categories, healthy class distribution (3.3x ratio — no resampling needed)
- Three amount tiers identified:
  - **Tier 1** ($52–$105): grocery_pos, gas_transport, grocery_net, entertainment — easy to classify
  - **Tier 2** ($32–$48): home, kids_pets, food_dining, health_fitness, personal_care — needs extra features
  - **Tier 3** ($6–$13): misc_pos, misc_net, shopping_net, shopping_pos, travel — hardest, heavy overlap
- gender has weak signal (grocery_net skews Female ~2:1)
- gas_transport is easiest class (std=$16, very tight distribution)
- travel is hardest class (std=$526, extreme outliers up to $11,872)

**Features we will use:** amt, log(amt), gender, city_pop, job  
**Target:** category (14 classes)

---

## Phase 3 — Preprocessing
**Goal:** Convert raw data into numbers the model can learn from

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print('All imports successful')

In [ ]:
# log1p is safer than log — handles $0 transactions without crashing
# log(0) = undefined, log1p(0) = log(1+0) = 0
df_clean['log_amt'] = np.log1p(df_clean['amt'])

# Apply same log transform to city_pop — also heavily skewed
df_clean['log_city_pop'] = np.log1p(df_clean['city_pop'])

print('Sample — amt vs log_amt:')
print(df_clean[['amt', 'log_amt']].head(5).round(3))

In [ ]:
# Encode gender: F=0, M=1
# Only 2 unique values — Label Encoding is safe here
le_gender = LabelEncoder()
df_clean['gender_encoded'] = le_gender.fit_transform(df_clean['gender'])

print('Gender encoding:')
for original, encoded in zip(le_gender.classes_, le_gender.transform(le_gender.classes_)):
    print(f'  {original} -> {encoded}')

In [ ]:
# 476 unique jobs — too many for one-hot encoding (adds 476 columns)
# Frequency encoding: replace each job with how many times it appears
# Common jobs get higher numbers, rare jobs get lower numbers
job_freq = df_clean['job'].value_counts()
df_clean['job_encoded'] = df_clean['job'].map(job_freq)

print('Unique jobs:', df_clean['job'].nunique())
print('Min frequency:', df_clean['job_encoded'].min())
print('Max frequency:', df_clean['job_encoded'].max())

In [ ]:
# Assemble final feature matrix
# X = inputs the model sees, y = what it predicts
feature_cols = ['amt', 'log_amt', 'gender_encoded', 'job_encoded', 'log_city_pop']

X = df_clean[feature_cols]
y = df_clean['category']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFeature matrix preview:')
X.head()

In [ ]:
# 80% train, 20% test
# stratify=y ensures all 14 categories appear in both splits proportionally
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler: fit on train only — prevents data leakage from test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train shape:', X_train_scaled.shape)
print('X_test shape: ', X_test_scaled.shape)
print('y_train shape:', y_train.shape)
print('y_test shape: ', y_test.shape)

### Phase 3 Complete
- Created log_amt and log_city_pop features (handles skewed distributions)
- Encoded gender using Label Encoding (F=0, M=1)
- Encoded job using Frequency Encoding (476 unique values → 1 column)
- Split: 40,000 train / 10,000 test (stratified)
- Scaled with StandardScaler — fit on train only

---

## Phase 4a — Classical ML Baseline
**Goal:** Establish a baseline before BERT. Three models trained on 5 tabular features.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

print('Imports done')

In [ ]:
# Logistic Regression: assigns weights to features, picks highest score
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_f1 = f1_score(y_test, lr_preds, average='macro')
print(f'Logistic Regression — Macro F1: {lr_f1:.4f}')

In [ ]:
# Naive Bayes: calculates probability per class, picks highest probability
print('Training Naive Bayes...')
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_preds = nb_model.predict(X_test_scaled)
nb_f1 = f1_score(y_test, nb_preds, average='macro')
print(f'Naive Bayes — Macro F1: {nb_f1:.4f}')

In [ ]:
# SVM: draws boundaries between every pair of classes, class winning most battles wins
print('Training SVM...')
svm_model = LinearSVC(max_iter=2000, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_preds = svm_model.predict(X_test_scaled)
svm_f1 = f1_score(y_test, svm_preds, average='macro')
print(f'SVM — Macro F1: {svm_f1:.4f}')

In [ ]:
# Summary comparison table
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'SVM'],
    'Macro F1': [round(lr_f1, 4), round(nb_f1, 4), round(svm_f1, 4)]
}).sort_values('Macro F1', ascending=False)

print(results_df.to_string(index=False))
print('\nReason for low scores: Tier 3 categories (5 classes) all have')
print('transaction amounts between $6-$13. Tabular features alone cannot')
print('separate them. BERT is needed to extract patterns from text.')

### Phase 4a Complete
All three classical models scored ~0.11 Macro F1. Root cause: Tier 3 categories share nearly identical amount ranges. These scores serve as our baseline — BERT's improvement over them tells the real story.

---

## Phase 4b — BERT Training (Ablation Experiment)

**Key design decision:** Merchant names in this dataset are anonymized (e.g., `fraud_Weber and Sons`). A naive row-level split produced F1=0.99 because the same merchant names appeared in both train and test — the model memorized names, not patterns.

**Fix applied:**
1. `GroupShuffleSplit` — ensures zero merchant overlap between train and test
2. Merchant names masked with `"unknown"` — forces model to learn from amount, gender, job, city_pop only

This is an **ablation experiment** — it proves whether the model learned genuine spending patterns or relied on merchant name memorization.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

# Reload full dataset for BERT pipeline
df_full = pd.read_csv('/kaggle/input/expense-classifier-data/transactions.csv', nrows=50000).copy()

# Encode labels
label_encoder = LabelEncoder()
df_full['label'] = label_encoder.fit_transform(df_full['category'])

# Mask merchant names — ablation experiment
# Forces model to predict using only: amount, gender, job, city_pop
df_full['input_text_no_merchant'] = (
    'merchant: unknown' +
    ' amount: '          + df_full['amt'].astype(str) +
    ' gender: '          + df_full['gender'].astype(str) +
    ' job: '             + df_full['job'].astype(str) +
    ' city population: ' + df_full['city_pop'].astype(str)
)

print('Sample masked input:')
print(df_full['input_text_no_merchant'].iloc[0])

In [ ]:
# GroupShuffleSplit — splits by merchant groups
# No merchant appears in both train and test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df_full, groups=df_full['merchant']))

train_df = df_full.iloc[train_idx].copy()
temp_df  = df_full.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['merchant']))

val_df  = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Assign labels to all splits
train_df['label'] = label_encoder.transform(train_df['category'])
val_df['label']   = label_encoder.transform(val_df['category'])
test_df['label']  = label_encoder.transform(test_df['category'])

print('Train rows:', train_df.shape[0])
print('Val rows  :', val_df.shape[0])
print('Test rows :', test_df.shape[0])

# Verify zero merchant overlap
print('\nMerchant overlap — Train∩Val :', len(set(train_df['merchant']) & set(val_df['merchant'])))
print('Merchant overlap — Train∩Test:', len(set(train_df['merchant']) & set(test_df['merchant'])))

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 64

train_enc = tokenizer(train_df['input_text_no_merchant'].tolist(),
                      truncation=True, padding=True, max_length=MAX_LEN)
val_enc   = tokenizer(val_df['input_text_no_merchant'].tolist(),
                      truncation=True, padding=True, max_length=MAX_LEN)
test_enc  = tokenizer(test_df['input_text_no_merchant'].tolist(),
                      truncation=True, padding=True, max_length=MAX_LEN)

print('Train tokenized:', len(train_enc['input_ids']))
print('Val tokenized  :', len(val_enc['input_ids']))
print('Test tokenized :', len(test_enc['input_ids']))

In [ ]:
import torch

class ExpenseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ExpenseDataset(train_enc, train_df['label'].tolist())
val_dataset   = ExpenseDataset(val_enc,   val_df['label'].tolist())
test_dataset  = ExpenseDataset(test_enc,  test_df['label'].tolist())

print('Train dataset:', len(train_dataset))
print('Val dataset  :', len(val_dataset))
print('Test dataset :', len(test_dataset))

In [ ]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT and add classification head for 14 categories
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(label_encoder.classes_)
)

print('BERT model loaded')
print('Number of output labels:', len(label_encoder.classes_))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score as sk_f1
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1'      : sk_f1(labels, predictions, average='macro')
    }

training_args = TrainingArguments(
    output_dir                  = './bert_no_merchant',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    learning_rate               = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 3,
    weight_decay                = 0.01,
    logging_steps               = 100,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1'
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)

print('Trainer ready')

In [ ]:
# Start training — takes ~20-30 minutes on Kaggle GPU T4
trainer.train()

---
## Phase 5 — Evaluation

In [ ]:
from sklearn.metrics import classification_report

# Evaluate on completely unseen test merchants
predictions = trainer.predict(test_dataset)
test_preds  = np.argmax(predictions.predictions, axis=1)
test_labels = predictions.label_ids

print(classification_report(
    test_labels,
    test_preds,
    target_names=label_encoder.classes_
))

In [ ]:
# Confusion matrix — shows which categories the model confuses most
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cmap='Blues')
plt.title('Confusion Matrix — BERT (No Merchant Names)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Phase 6 — Live Prediction Demo

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def predict_expense(amount, gender, job, city_pop):
    """
    Predicts expense category for a new transaction.
    Merchant name is masked — prediction based on spending features only.
    """
    text = (
        f'merchant: unknown'
        f' amount: {amount}'
        f' gender: {gender}'
        f' job: {job}'
        f' city population: {city_pop}'
    )

    inputs = tokenizer(text, return_tensors='pt',
                       truncation=True, max_length=64)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred_idx   = outputs.logits.argmax().item()
    pred_label = label_encoder.classes_[pred_idx]
    confidence = torch.softmax(outputs.logits, dim=1).max().item()

    print(f'Input     : amount=${amount}, gender={gender}, job={job}, city_pop={city_pop}')
    print(f'Predicted : {pred_label}')
    print(f'Confidence: {confidence:.2%}')
    print()

# Sample predictions
predict_expense(amount=4.99,  gender='F', job='Teacher',  city_pop=50000)
predict_expense(amount=105.0, gender='M', job='Engineer', city_pop=800000)
predict_expense(amount=6.0,   gender='M', job='Lawyer',   city_pop=500000)

In [ ]:
# Save final model
import json

trainer.save_model('./bert_expense_final')

with open('./bert_expense_final/label_classes.json', 'w') as f:
    json.dump(label_encoder.classes_.tolist(), f)

print('Model saved to ./bert_expense_final')
print('Label classes saved to ./bert_expense_final/label_classes.json')

---
## Results Summary

| Experiment | Setup | Macro F1 | Finding |
|---|---|---|---|
| Classical ML (LR, NB, SVM) | Tabular features | ~0.11 | Tier 3 overlap — tabular insufficient |
| BERT (with merchant names) | Row-level split | ~0.99 | Artificially high — merchant names memorized |
| BERT (no merchant names) | Merchant-group split | ~0.17 | Honest score — 82% of signal was merchant name |

**Conclusion:** Merchant name patterns carried ~82% of predictive signal. With anonymized merchants, the model learns some signal from amount, job and city_pop (F1 rises from 0.11 to 0.17) but cannot generalize strongly. A dataset with real merchant names and MCC codes would unlock the full potential of BERT.